# Capstone Chapter 4: Outpatient Queue Simulation & Comparative Analysis
## Heart Check PHC: A Kiosk-Based Queue Management and Analytics System
**Philippine Heart Center — Outpatient Department (OPD)**  
**STI College Caloocan — BS Computer Science**  
**Researchers:** Justin Dayle D. Caasi, Reign Daniel C. Gutierrez, Lenardo R. Jualo, Jensen B. Urrutia  
**Last Updated:** 2026-10-19  

---

### Non-Invasive Research Notice: ZERO Database Insertion
This Jupyter Notebook is designed strictly for **academic and analytical evaluation**.  
- **Zero Database Operations:** Absolutely no database connections, mutations, or table insertions are performed.
- **Local Empirical Grounding:** Raw clinical stage records are extracted in-memory directly from the physical hospital consultation room Excel tracking logs (`python_backend/data_entries/2024_JAN.xls` through `2024_DEC.xls`).
- **Dataset Scope:** The **whole year of 2024 (January 1, 2024 to December 31, 2024, 12 full calendar months)**, encompassing **26,753 validated patient visits**.
- **Scoping to Consultation:** Scoped to the Consultation service, representing approximately 45% of total OPD attendance and the sole service with real timestamped tracking records.


### Research Question & Methodology Overview

> **Primary Research Question:**  
> "Does replacing the traditional manual pen-and-paper OPD queue with the Heart Check PHC automated digital queue system reduce patient waiting time at the registration desk, and by how much?"

#### Compared Scenarios:
1. **Scenario A (Traditional Manual Paper-Based Queue):**
   - Patient arrives at OPD.
   - Staff writes patient name in physical logbook and issues a handwritten queue stub.
   - Staff calls patient name verbally or via public address (PA) system.
   - Registration desk clerk manually verifies appointment and enters details into paper ledger.
   - Clinical stages (consultation wait, physician consultation, carryout) proceed as recorded.
2. **Scenario B (Heart Check PHC Automated Digital Queue):**
   - Patient self-registers at the touchscreen kiosk (~40 seconds).
   - Thermal ticket is issued automatically with queue number.
   - High-visibility public waiting room monitors and automated Text-to-Speech (TTS) callouts notify the patient (eliminating verbal announcement latency).
   - Registration clerk views pre-populated digital records, reducing desk data-entry overhead.
   - Identical clinical stages proceed with constant physician interaction duration.


In [ ]:
%load_ext autoreload
%autoreload

import os
import sys
import glob
import re
from datetime import datetime, date, timedelta
import numpy as np
import pandas as pd

# Safe display import for Jupyter and standalone environments
try:
    from IPython.display import display
except ImportError:
    display = print

# Robust directory discovery whether the kernel is started in repo root or python_backend
current_dir = os.getcwd()
if os.path.basename(current_dir) == "python_backend":
    BACKEND_DIR = current_dir
elif os.path.exists(os.path.join(current_dir, "python_backend")):
    BACKEND_DIR = os.path.join(current_dir, "python_backend")
else:
    BACKEND_DIR = current_dir

if BACKEND_DIR not in sys.path:
    sys.path.insert(0, BACKEND_DIR)

# Import existing project analytics helpers and constants
from analytics.helpers import mm1, arrival_rate, service_rate, to_utc
from analytics.preprocessing import preprocess_queue_data
from analytics.constants import (
    OPD_TARGET_MINUTES,
    PHC_EXCEL_WAITING_TIME_TARGET_MINUTES,
    OVERWHELMED_MINUTES,
)

# Set pandas display options for clean tabular inspection
pd.set_option('display.max_columns', 15)
pd.set_option('display.width', 1000)

DATA_DIR = os.path.join(BACKEND_DIR, "data_entries")
OUTPUT_DIR = os.path.join(BACKEND_DIR, "output")
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"Environment initialized successfully.")
print(f"Backend directory: {BACKEND_DIR}")
print(f"Data directory:    {DATA_DIR}")
print(f"Output directory:  {OUTPUT_DIR}")
print("Analytics modules loaded from local repository. Zero database dependencies.")


In [ ]:
# ---------------------------------------------------------------------------
# Operational Parameters & Observational Overhead Constants
# ---------------------------------------------------------------------------

# Scenario A: Traditional manual queue pre-desk overhead (observed at PHC OPD)
MANUAL_STUB_MEAN_MIN = 3.5      # Staff writes name in ledger and cuts stub
MANUAL_STUB_STD_MIN = 0.8       # Standard deviation of manual stub issuance
MANUAL_STUB_FLOOR_MIN = 0.5     # Minimum threshold for physical stub issuance

MANUAL_ANNOUNCE_MEAN_MIN = 1.5  # Verbal PA announcement lag and acoustic latency
MANUAL_ANNOUNCE_STD_MIN = 0.3   # Standard deviation of verbal callout delay
MANUAL_ANNOUNCE_FLOOR_MIN = 0.0

# Scenario B: Automated digital queue parameters
KIOSK_CHECKIN_MEAN_MIN = 0.67   # Mean kiosk touchscreen duration (~40 seconds)
KIOSK_CHECKIN_STD_MIN = 0.15    # Standard deviation of kiosk interaction
KIOSK_CHECKIN_FLOOR_MIN = 0.1   # Minimum threshold for kiosk check-in

DIGITAL_ANNOUNCE_MIN = 0.0      # Eliminated by digital monitors and TTS callouts
DEFAULT_REG_REDUCTION_MIN = 2.0 # Time saved at desk via kiosk pre-filled records
DIGITAL_REG_FLOOR_MIN = 1.0     # Minimum essential clinical interaction duration

# Sensitivity analysis parameter variations (minutes)
SENSITIVITY_REDUCTIONS = [1.0, 2.0, 3.0]

print("Research overhead parameters configured.")


### Pure Local Excel Parser Functions
These parsing functions operate completely in-memory with **zero external database dependencies**:
1. `resolve_sheet_date`: Intelligently reconciles date headers with sheet names and workbook filenames.
2. `fix_time_heuristic`: Applies PHC clinic AM/PM conventions (hours 1:00 to 6:59 correspond to afternoon clinic hours 13:00 to 18:59, while 7:00 AM is preserved for early morning queue arrivals).


In [ ]:
def resolve_sheet_date(raw_cell_val, sheet_name: str, filename: str = "") -> str | None:
    base_date = date(1899, 12, 30)
    cell_date = None

    if isinstance(raw_cell_val, (int, float)) and raw_cell_val > 1000:
        cell_date = base_date + timedelta(days=int(raw_cell_val))
    elif isinstance(raw_cell_val, str) and raw_cell_val.strip():
        s = raw_cell_val.strip()
        m_typo = re.match(r"^(\d{2})(\d{2})/(\d{4})$", s)
        if m_typo:
            s = f"{m_typo.group(1)}/{m_typo.group(2)}/{m_typo.group(3)}"
        p = pd.to_datetime(s, errors="coerce")
        if pd.notna(p):
            cell_date = p.date()

    m_yr = re.search(r"(202\d)", filename) if filename else None
    file_year = int(m_yr.group(1)) if m_yr else None

    m_md = re.match(r"^(\d{1,2})[-/](\d{1,2})$", sheet_name.strip())
    if m_md:
        s_m, s_d = int(m_md.group(1)), int(m_md.group(2))
        target_year = file_year or (cell_date.year if cell_date else None)
        if target_year:
            return f"{target_year:04d}-{s_m:02d}-{s_d:02d}"

    if cell_date:
        yr = file_year or cell_date.year
        return f"{yr:04d}-{cell_date.month:02d}-{cell_date.day:02d}"

    return None

def fix_time_heuristic(t) -> str | None:
    if pd.isna(t):
        return None
    if hasattr(t, "hour"):
        hour = t.hour
        if 1 <= hour <= 6:
            hour += 12
        minute = getattr(t, "minute", 0)
        second = getattr(t, "second", 0)
        return f"{hour:02d}:{minute:02d}:{second:02d}"
    if isinstance(t, str):
        cleaned = t.strip()
        if not cleaned:
            return None
        match = re.match(r"^(\d{1,2}):(\d{2})(?::(\d{2}))?", cleaned)
        if match:
            hour = int(match.group(1))
            minute = int(match.group(2))
            second = int(match.group(3)) if match.group(3) else 0
            if 1 <= hour <= 6:
                hour += 12
            return f"{hour:02d}:{minute:02d}:{second:02d}"
    return None

print("Local parser utilities defined successfully.")


### In-Memory Extraction of 2024 PHC Workbooks
Extracts all patient records across the 12 workbooks (`2024_JAN.xls` through `2024_DEC.xls`).


In [ ]:
def load_all_2024_excel_records() -> pd.DataFrame:
    excel_files = sorted(glob.glob(os.path.join(DATA_DIR, "2024_*.xls")))

    if not excel_files:
        raise FileNotFoundError(f"No 2024 Excel files found in {DATA_DIR}.")

    print(f"Discovered {len(excel_files)} monthly hospital workbooks for year 2024:")
    all_dfs = []

    for path in excel_files:
        filename = os.path.basename(path)
        try:
            xls = pd.ExcelFile(path, engine="xlrd")
        except Exception as exc:
            print(f"Warning: Could not open {filename}: {exc}")
            continue

        file_dfs = []
        for sheet_name in xls.sheet_names:
            try:
                raw = pd.read_excel(path, sheet_name=sheet_name, header=None, engine="xlrd")
            except Exception:
                continue

            date_val = None
            for r in range(min(10, len(raw))):
                row_vals = raw.iloc[r].astype(str)
                if row_vals.str.contains("Date:").any():
                    col_idx = row_vals[row_vals.str.contains("Date:")].index[0]
                    date_val = raw.iloc[r, col_idx + 1]
                    break

            header_row, hosp_col = None, None
            for r in range(min(15, len(raw))):
                row_vals = raw.iloc[r].astype(str)
                matches = row_vals[row_vals.str.contains("Hospital", na=False)]
                matches = matches[matches.index >= 15]
                if len(matches) > 0:
                    header_row = r
                    hosp_col = matches.index[0]
                    break

            if header_row is None:
                continue

            data_start = header_row + 2
            rows = []
            for r in range(data_start, len(raw)):
                hosp_num = raw.iloc[r, hosp_col]
                hosp_str = str(hosp_num).strip().replace(".0", "")
                if pd.isna(hosp_num) or not hosp_str.isdigit() or hosp_str == "0":
                    break
                rows.append({
                    "patientNum": hosp_str,
                    "reg_start": raw.iloc[r, hosp_col + 1],
                    "reg_end": raw.iloc[r, hosp_col + 2],
                    "consult_start": raw.iloc[r, hosp_col + 3],
                    "consult_end": raw.iloc[r, hosp_col + 4],
                    "carryout_end": raw.iloc[r, hosp_col + 5] if (hosp_col + 5) < len(raw.columns) else None,
                })

            if not rows:
                continue

            resolved_date = resolve_sheet_date(date_val, sheet_name, filename)
            sheet_df = pd.DataFrame(rows)
            sheet_df["sheet_date"] = resolved_date
            sheet_df["sheet_name"] = sheet_name
            file_dfs.append(sheet_df)

        if file_dfs:
            combined_file = pd.concat(file_dfs, ignore_index=True)
            print(f"  Extracted {len(combined_file):>5} patient rows from {filename}")
            all_dfs.append(combined_file)

    full_df = pd.concat(all_dfs, ignore_index=True)

    # Convert timestamps with clinic AM/PM correction
    date_part = pd.to_datetime(full_df["sheet_date"], errors="coerce").dt.date.astype(str)

    for col in ["reg_start", "reg_end", "consult_start", "consult_end", "carryout_end"]:
        if col in full_df.columns:
            time_str = full_df[col].apply(fix_time_heuristic)
            combined = pd.to_datetime(date_part + " " + time_str, errors="coerce")
            full_df[col] = (
                combined.dt.tz_localize("Asia/Manila", ambiguous="NaT", nonexistent="NaT")
                .dt.tz_convert("UTC")
            )

    required = ["reg_start", "reg_end", "consult_start", "consult_end"]
    full_df = full_df.dropna(subset=required)

    full_df["created_at"] = full_df["reg_start"]
    full_df["service"] = "Consultation"
    full_df["status"] = "Done"
    full_df["is_historical"] = True
    full_df["cubicleNum"] = None
    full_df["carryout_start"] = full_df["consult_end"].where(full_df["carryout_end"].notna())

    # Preprocess duration metrics via standard pipeline
    df_clean = preprocess_queue_data(full_df)

    # Filter validated rows
    clean_mask = (
        df_clean["service_registration"].notna()
        & (df_clean["service_registration"] > 0)
        & df_clean["service_consultation"].notna()
        & (df_clean["service_consultation"] > 0)
        & df_clean["wait_consultation"].notna()
        & (df_clean["wait_consultation"] >= 0)
    )
    df_final = df_clean[clean_mask].copy()

    if "visit_date" in df_final.columns:
        df_final["visit_year"] = pd.to_datetime(df_final["visit_date"]).dt.year
        df_final = df_final[df_final["visit_year"] == 2024]

    print(f"\nExtraction complete: {len(df_final):,} validated patient visits across whole year 2024.")
    return df_final

df_raw = load_all_2024_excel_records()


## Section 1: Empirical 2024 Baseline Descriptives (Grand Mean)

### Rationale for the Grand Mean (Pooled Average)
In queueing theory ($M/M/1$), the true service rate $\mu$ of a service counter is defined as total customers served divided by total service time across the entire evaluation horizon.  
Using the **Grand Mean** gives every patient equal weight ($1/N$), preventing months with thin sample sizes (e.g., May with fewer clinic days) from distorting the operational baseline.


In [ ]:
def extract_baseline_metrics(df: pd.DataFrame) -> dict:
    def safe_stats(series: pd.Series) -> dict:
        clean = series.dropna()
        clean = clean[clean > 0]
        return {
            "mean": round(float(clean.mean()), 2),
            "std": round(float(clean.std()), 2) if len(clean) > 1 else 0.0,
            "median": round(float(clean.median()), 2),
            "min": round(float(clean.min()), 2),
            "max": round(float(clean.max()), 2),
            "n": int(len(clean)),
        }

    metrics = {
        "service_registration": safe_stats(df["service_registration"]),
        "wait_consultation": safe_stats(df["wait_consultation"]),
        "service_consultation": safe_stats(df["service_consultation"]),
        "total_time": safe_stats(df["total_time"]),
    }

    if "service_carryout" in df.columns:
        metrics["service_carryout"] = safe_stats(df["service_carryout"])
    else:
        metrics["service_carryout"] = {"mean": 0.0, "std": 0.0, "median": 0.0, "n": 0}

    arrival_col = "reg_start" if "reg_start" in df.columns else "kiosk_time"
    metrics["full_day_lambda"] = arrival_rate(df[arrival_col])

    # Morning peak window (08:00 to 12:00 Manila time)
    peak_df = df[df["hour"].isin([8, 9, 10, 11])] if "hour" in df.columns else df
    metrics["peak_lambda"] = arrival_rate(peak_df[arrival_col])
    metrics["peak_n"] = len(peak_df)

    return metrics

baseline = extract_baseline_metrics(df_raw)

baseline_summary = pd.DataFrame([
    {
        "Clinical Stage": "Registration Desk Service Duration",
        "Mean (min)": baseline["service_registration"]["mean"],
        "Std Dev (min)": baseline["service_registration"]["std"],
        "Median (min)": baseline["service_registration"]["median"],
        "Sample Size (n)": baseline["service_registration"]["n"],
    },
    {
        "Clinical Stage": "Wait for Doctor Consultation",
        "Mean (min)": baseline["wait_consultation"]["mean"],
        "Std Dev (min)": baseline["wait_consultation"]["std"],
        "Median (min)": baseline["wait_consultation"]["median"],
        "Sample Size (n)": baseline["wait_consultation"]["n"],
    },
    {
        "Clinical Stage": "Physician Consultation Duration",
        "Mean (min)": baseline["service_consultation"]["mean"],
        "Std Dev (min)": baseline["service_consultation"]["std"],
        "Median (min)": baseline["service_consultation"]["median"],
        "Sample Size (n)": baseline["service_consultation"]["n"],
    },
    {
        "Clinical Stage": "Carry-Out Processing Duration",
        "Mean (min)": baseline["service_carryout"]["mean"],
        "Std Dev (min)": baseline["service_carryout"]["std"],
        "Median (min)": baseline["service_carryout"]["median"],
        "Sample Size (n)": baseline["service_carryout"]["n"],
    },
])

print(f"Full-Day Arrival Rate (lambda):          {baseline['full_day_lambda']:.4f} patients/min")
print(f"Peak Morning Arrival Rate (08:00-12:00):  {baseline['peak_lambda']:.4f} patients/min (n = {baseline['peak_n']:,})")
display(baseline_summary)


## Section 2: Reconstructing Comparative Scenarios (A vs. B)

- **Scenario A (Manual Paper-Based Queue):**
  - Stub issuance overhead: $\mathcal{N}(\mu = 3.5, \sigma = 0.8)$, floor $0.5$ min (derived from staff interviews and clinic observation).
  - Verbal PA announcement latency: $\mathcal{N}(\mu = 1.5, \sigma = 0.3)$, floor $0.0$ min.
  - Registration desk duration: Empirical historical duration ($31.63$ min mean).
- **Scenario B (Heart Check PHC Automated Digital Queue):**
  - Kiosk self-registration: $\mathcal{N}(\mu = 0.67, \sigma = 0.15)$, floor $0.1$ min (~40 seconds).
  - Announcement delay: $0.0$ min (eliminated by real-time waiting room displays and automated TTS callouts).
  - Accelerated registration desk: Reduced by kiosk pre-fill benefit ($2.0$ min reduction), bounded by a minimum clinical floor of $1.0$ min.
- **Controlled Clinical Variable:** Wait for consultation, consultation duration, and carryout duration remain identical in both scenarios.


In [ ]:
def add_manual_overhead(df: pd.DataFrame, seed: int = 42) -> pd.DataFrame:
    rng = np.random.default_rng(seed)
    n = len(df)
    res = df.copy()

    res["stub_issuance_min"] = np.clip(
        rng.normal(MANUAL_STUB_MEAN_MIN, MANUAL_STUB_STD_MIN, n),
        MANUAL_STUB_FLOOR_MIN, None
    )
    res["announcement_min"] = np.clip(
        rng.normal(MANUAL_ANNOUNCE_MEAN_MIN, MANUAL_ANNOUNCE_STD_MIN, n),
        MANUAL_ANNOUNCE_FLOOR_MIN, None
    )
    res["service_registration_sim"] = res["service_registration"]

    carryout_val = res["service_carryout"].fillna(0) if "service_carryout" in res.columns else 0.0

    res["total_opd_sim"] = (
        res["stub_issuance_min"]
        + res["announcement_min"]
        + res["service_registration_sim"].fillna(0)
        + res["wait_consultation"].fillna(0)
        + res["service_consultation"].fillna(0)
        + carryout_val
    )
    return res

def add_digital_overhead(df: pd.DataFrame, reg_reduction_min: float = 2.0, seed: int = 42) -> pd.DataFrame:
    rng = np.random.default_rng(seed)
    n = len(df)
    res = df.copy()

    res["stub_issuance_min"] = np.clip(
        rng.normal(KIOSK_CHECKIN_MEAN_MIN, KIOSK_CHECKIN_STD_MIN, n),
        KIOSK_CHECKIN_FLOOR_MIN, None
    )
    res["announcement_min"] = DIGITAL_ANNOUNCE_MIN

    res["service_registration_sim"] = np.clip(
        res["service_registration"] - reg_reduction_min,
        DIGITAL_REG_FLOOR_MIN, None
    )

    carryout_val = res["service_carryout"].fillna(0) if "service_carryout" in res.columns else 0.0

    res["total_opd_sim"] = (
        res["stub_issuance_min"]
        + res["announcement_min"]
        + res["service_registration_sim"].fillna(0)
        + res["wait_consultation"].fillna(0)
        + res["service_consultation"].fillna(0)
        + carryout_val
    )
    return res

df_scenario_a = add_manual_overhead(df_raw, seed=42)
df_scenario_b = add_digital_overhead(df_raw, reg_reduction_min=2.0, seed=42)

print("Scenario A (Manual) and Scenario B (Digital) successfully reconstructed.")


## Section 3: Stage-by-Stage Comparative Analysis
Consolidates per-stage durations across a patient's outpatient journey under each scenario.


In [ ]:
def build_stage_comparison_table(df_a: pd.DataFrame, df_b: pd.DataFrame, baseline: dict) -> pd.DataFrame:
    stub_a = round(float(df_a["stub_issuance_min"].mean()), 2)
    stub_b = round(float(df_b["stub_issuance_min"].mean()), 2)
    ann_a = round(float(df_a["announcement_min"].mean()), 2)
    ann_b = 0.0
    reg_a = round(float(df_a["service_registration_sim"].mean()), 2)
    reg_b = round(float(df_b["service_registration_sim"].mean()), 2)

    wait_consult = baseline["wait_consultation"]["mean"]
    serv_consult = baseline["service_consultation"]["mean"]
    carryout = baseline.get("service_carryout", {}).get("mean", 0.0)

    total_a = round(float(df_a["total_opd_sim"].mean()), 2)
    total_b = round(float(df_b["total_opd_sim"].mean()), 2)

    stages = [
        ("Ticket / Queue Stub Acquisition", stub_a, stub_b),
        ("Queue Announcement Latency", ann_a, ann_b),
        ("Registration Counter Service", reg_a, reg_b),
        ("Wait for Consultation (Empirical)", wait_consult, wait_consult),
        ("Consultation Service (Empirical)", serv_consult, serv_consult),
        ("Carry-Out Service (Empirical)", carryout, carryout),
        ("TOTAL OPD STAY (Simulated Mean)", total_a, total_b),
    ]

    rows = []
    for stage_name, val_a, val_b in stages:
        saved = round(val_a - val_b, 2)
        pct = round(saved / val_a * 100, 1) if val_a > 0 else 0.0
        rows.append({
            "Stage": stage_name,
            "Manual Scenario A (min)": val_a,
            "Digital Scenario B (min)": val_b,
            "Time Saved (min)": saved,
            "Reduction (%)": pct,
        })
    return pd.DataFrame(rows)

stage_comparison_table = build_stage_comparison_table(df_scenario_a, df_scenario_b, baseline)
display(stage_comparison_table)


## Section 4: Registration Counter $M/M/1$ Queueing Model (Peak Morning Window)

### Queue Formulation
The registration desk is modeled as a single-server queue ($c = 1$) operating under First-In, First-Out (FIFO) discipline.
$$\lambda = \text{Morning Peak Arrival Rate (patients/min)}, \quad \mu = \frac{1}{\text{Mean Registration Service Duration}}$$
$$\rho = \frac{\lambda}{\mu}, \quad L_q = \frac{\rho^2}{1 - \rho}, \quad W_q = \frac{L_q}{\lambda}, \quad W = \frac{1}{\mu - \lambda}$$


In [ ]:
peak_lambda = baseline["peak_lambda"]
reg_mean_a = baseline["service_registration"]["mean"]
reg_mean_b = float(df_scenario_b["service_registration_sim"].mean())

mu_a = 1.0 / reg_mean_a
mu_b = 1.0 / reg_mean_b

mm1_a = mm1(peak_lambda, mu_a)
mm1_b = mm1(peak_lambda, mu_b)

def safe_fmt(val, fmt="{:.2f}"):
    return fmt.format(val) if val is not None and not isinstance(val, str) else str(val)

mm1_table = pd.DataFrame([
    {
        "M/M/1 Metric": "Arrival Rate lambda (patients/min)",
        "Scenario A (Manual)": f"{peak_lambda:.4f}",
        "Scenario B (Digital)": f"{peak_lambda:.4f}",
        "Benefit / Improvement": "Equal morning arrival surge",
    },
    {
        "M/M/1 Metric": "Service Rate mu (patients/min)",
        "Scenario A (Manual)": f"{mu_a:.4f}",
        "Scenario B (Digital)": f"{mu_b:.4f}",
        "Benefit / Improvement": f"+{((mu_b - mu_a) / mu_a * 100):.1f}% clerk processing capacity",
    },
    {
        "M/M/1 Metric": "Traffic Intensity (rho)",
        "Scenario A (Manual)": safe_fmt(mm1_a.get("utilization_rho"), "{:.4f}"),
        "Scenario B (Digital)": safe_fmt(mm1_b.get("utilization_rho"), "{:.4f}"),
        "Benefit / Improvement": f"-{round(mm1_a.get('utilization_rho', 0) - mm1_b.get('utilization_rho', 0), 4):.4f} server utilization reduction",
    },
    {
        "M/M/1 Metric": "Average Patients in Queue (Lq)",
        "Scenario A (Manual)": safe_fmt(mm1_a.get("avg_in_queue_Lq"), "{:.4f}"),
        "Scenario B (Digital)": safe_fmt(mm1_b.get("avg_in_queue_Lq"), "{:.4f}"),
        "Benefit / Improvement": "-25.8% fewer queueing patients",
    },
    {
        "M/M/1 Metric": "Average Queue Waiting Time (Wq)",
        "Scenario A (Manual)": f"{safe_fmt(mm1_a.get('avg_wait_queue_Wq_min'))} min",
        "Scenario B (Digital)": f"{safe_fmt(mm1_b.get('avg_wait_queue_Wq_min'))} min",
        "Benefit / Improvement": f"-{round(mm1_a.get('avg_wait_queue_Wq_min', 0) - mm1_b.get('avg_wait_queue_Wq_min', 0), 2):.2f} min saved in registration wait",
    },
    {
        "M/M/1 Metric": "Total Time in Registration Subsystem (W)",
        "Scenario A (Manual)": f"{safe_fmt(mm1_a.get('avg_time_system_W_min'))} min",
        "Scenario B (Digital)": f"{safe_fmt(mm1_b.get('avg_time_system_W_min'))} min",
        "Benefit / Improvement": f"-{round(mm1_a.get('avg_time_system_W_min', 0) - mm1_b.get('avg_time_system_W_min', 0), 2):.2f} min total subsystem time reduction",
    },
])

display(mm1_table)


## Section 5: Hospital OPD Compliance Targets
Evaluates compliance against official hospital targets:
- **150-Minute Standard Target:** Standard PHC outpatient duration (`OPD_TARGET_MINUTES`).
- **180-Minute Upper Ceiling:** Maximum operational ceiling (`PHC_EXCEL_WAITING_TIME_TARGET_MINUTES`).
- **Peak Morning Hour (08:00):** Compliance for patients arriving during the peak opening surge.


In [ ]:
def compute_compliance_rates(df: pd.DataFrame) -> dict:
    total = len(df)
    within_150 = (df["total_opd_sim"] <= OPD_TARGET_MINUTES).sum()
    within_180 = (df["total_opd_sim"] <= PHC_EXCEL_WAITING_TIME_TARGET_MINUTES).sum()

    peak_df = df[df["hour"] == 8] if "hour" in df.columns else pd.DataFrame()
    peak_n = len(peak_df)
    peak_within_150 = (peak_df["total_opd_sim"] <= OPD_TARGET_MINUTES).sum() if peak_n > 0 else 0

    return {
        "total": total,
        "within_150_pct": round(float(within_150 / total * 100), 2),
        "within_150_n": int(within_150),
        "within_180_pct": round(float(within_180 / total * 100), 2),
        "within_180_n": int(within_180),
        "peak_within_150_pct": round(float(peak_within_150 / peak_n * 100), 2) if peak_n > 0 else 0.0,
        "peak_n": peak_n,
        "peak_within_150_n": int(peak_within_150),
    }

comp_a = compute_compliance_rates(df_scenario_a)
comp_b = compute_compliance_rates(df_scenario_b)

compliance_table = pd.DataFrame([
    {
        "Compliance Standard": "150-Minute OPD Target",
        "Scenario A (Manual)": f"{comp_a['within_150_pct']}% ({comp_a['within_150_n']:,} pts)",
        "Scenario B (Digital)": f"{comp_b['within_150_pct']}% ({comp_b['within_150_n']:,} pts)",
        "Absolute Improvement": f"+{round(comp_b['within_150_pct'] - comp_a['within_150_pct'], 2)}% pts (+{comp_b['within_150_n'] - comp_a['within_150_n']:,} patients)",
    },
    {
        "Compliance Standard": "180-Minute Maximum Ceiling",
        "Scenario A (Manual)": f"{comp_a['within_180_pct']}% ({comp_a['within_180_n']:,} pts)",
        "Scenario B (Digital)": f"{comp_b['within_180_pct']}% ({comp_b['within_180_n']:,} pts)",
        "Absolute Improvement": f"+{round(comp_b['within_180_pct'] - comp_a['within_180_pct'], 2)}% pts (+{comp_b['within_180_n'] - comp_a['within_180_n']:,} patients)",
    },
    {
        "Compliance Standard": "Peak Morning Hour (08:00) 150-Min",
        "Scenario A (Manual)": f"{comp_a['peak_within_150_pct']}% ({comp_a['peak_within_150_n']:,} pts)",
        "Scenario B (Digital)": f"{comp_b['peak_within_150_pct']}% ({comp_b['peak_within_150_n']:,} pts)",
        "Absolute Improvement": f"+{round(comp_b['peak_within_150_pct'] - comp_a['peak_within_150_pct'], 2)}% pts (+{comp_b['peak_within_150_n'] - comp_a['peak_within_150_n']:,} patients)",
    },
])

display(compliance_table)


## Section 6: Sensitivity Analysis Across Desk Reductions
Tests model robustness across Conservative ($1.0$ min), Base Case ($2.0$ min), and Optimistic ($3.0$ min) desk time savings.


In [ ]:
def run_sensitivity_analysis(df: pd.DataFrame, peak_lambda: float) -> pd.DataFrame:
    rows = []
    df_man = add_manual_overhead(df, seed=42)
    manual_total_mean = float(df_man["total_opd_sim"].mean())

    for red in SENSITIVITY_REDUCTIONS:
        df_dig = add_digital_overhead(df, reg_reduction_min=red, seed=42)
        dig_total_mean = float(df_dig["total_opd_sim"].mean())
        saved_min = round(manual_total_mean - dig_total_mean, 2)
        saved_pct = round(saved_min / manual_total_mean * 100, 1)

        dig_reg_mean = float(df_dig["service_registration_sim"].mean())
        dig_mu = 1.0 / dig_reg_mean
        dig_m = mm1(peak_lambda, dig_mu)

        comp = compute_compliance_rates(df_dig)
        label = "Conservative (1.0 min)" if red == 1.0 else "Base Case (2.0 min)" if red == 2.0 else "Optimistic (3.0 min)"

        rows.append({
            "Reduction Model": label,
            "Desk Reduction": f"{red:.1f} min",
            "Effective Reg Time": f"{dig_reg_mean:.2f} min",
            "Traffic Intensity (rho)": dig_m.get("utilization_rho"),
            "Queue Wait Wq (min)": dig_m.get("avg_wait_queue_Wq_min"),
            "Total OPD Stay (min)": round(dig_total_mean, 2),
            "Time Saved (min)": saved_min,
            "Total Stay Reduction (%)": saved_pct,
            "150-Min Compliance (%)": comp["within_150_pct"],
        })
    return pd.DataFrame(rows)

sensitivity_table = run_sensitivity_analysis(df_raw, peak_lambda)
display(sensitivity_table)


## Section 7: Monthly Continuity Verification (12 Calendar Months of 2024)
Confirms complete temporal coverage and stability across all 12 months of 2024.


In [ ]:
def compute_monthly_breakdown(df: pd.DataFrame) -> pd.DataFrame:
    df_copy = df.copy()
    df_copy["month_num"] = pd.to_datetime(df_copy["visit_date"]).dt.month
    month_names = {
        1: "January", 2: "February", 3: "March", 4: "April",
        5: "May", 6: "June", 7: "July", 8: "August",
        9: "September", 10: "October", 11: "November", 12: "December"
    }

    rows = []
    for m in range(1, 13):
        m_df = df_copy[df_copy["month_num"] == m]
        if m_df.empty:
            continue
        rows.append({
            "Month": month_names.get(m, f"Month {m}"),
            "Patients": int(len(m_df)),
            "Mean Reg (min)": round(float(m_df["service_registration"].mean()), 2),
            "Mean Consult Wait (min)": round(float(m_df["wait_consultation"].mean()), 2),
            "Mean Consult Service (min)": round(float(m_df["service_consultation"].mean()), 2),
            "Mean Total Stay (min)": round(float(m_df["total_time"].mean()), 2),
        })
    return pd.DataFrame(rows)

monthly_breakdown_table = compute_monthly_breakdown(df_raw)
display(monthly_breakdown_table)


## Section 8: Exporting Publication-Ready CSV Appendices
Exports clean tabular datasets to `output/` for direct inclusion in thesis document tables and appendices.


In [ ]:
# Export all 7 tabular artifacts to output/
stage_comparison_table.to_csv(os.path.join(OUTPUT_DIR, "chapter4_stage_comparison.csv"), index=False)
mm1_table.to_csv(os.path.join(OUTPUT_DIR, "chapter4_mm1_metrics.csv"), index=False)
compliance_table.to_csv(os.path.join(OUTPUT_DIR, "chapter4_compliance_summary.csv"), index=False)
sensitivity_table.to_csv(os.path.join(OUTPUT_DIR, "chapter4_sensitivity_analysis.csv"), index=False)
monthly_breakdown_table.to_csv(os.path.join(OUTPUT_DIR, "chapter4_monthly_breakdown_2024.csv"), index=False)

df_scenario_a[["stub_issuance_min", "announcement_min", "service_registration_sim", "wait_consultation", "service_consultation", "total_opd_sim"]].describe().round(2).to_csv(os.path.join(OUTPUT_DIR, "chapter4_scenario_a_patient_stats.csv"))
df_scenario_b[["stub_issuance_min", "announcement_min", "service_registration_sim", "wait_consultation", "service_consultation", "total_opd_sim"]].describe().round(2).to_csv(os.path.join(OUTPUT_DIR, "chapter4_scenario_b_patient_stats.csv"))

print("Exported 7 thesis CSV datasets to:", OUTPUT_DIR)
for f in sorted(glob.glob(os.path.join(OUTPUT_DIR, "chapter4_*.csv"))):
    print(f"  - {os.path.basename(f)} ({os.path.getsize(f)} bytes)")


## Section 9: Executive Thesis Summary & Defense Key Findings

### Primary Manuscript Findings:
1. **Total OPD Stay Reduction:**
   > Under empirical evaluation of 26,753 consultation visits throughout the whole year of 2024, the Heart Check PHC automated digital queue system reduces average total outpatient visit duration from **149.36 minutes** (Scenario A) to **143.04 minutes** (Scenario B), achieving a mean saving of **6.32 minutes per patient (4.2% reduction)**. This improvement is driven by eliminating manual logbook stub issuance (saving **2.84 minutes**, an 80.9% reduction), eliminating acoustic callout latency (saving **1.50 minutes**, a 100% reduction), and accelerating registration desk verification (saving **1.98 minutes**).

2. **Registration Desk Queue Stabilization:**
   > During peak morning intake (08:00 to 12:00 Manila time), kiosk pre-registration increases registration clerk service capacity from $\mu = 0.0316$ to $\mu = 0.0337$ patients/min. This reduces traffic intensity from $\rho = 0.7465$ down to $\rho = 0.6997$ and decreases average queue waiting time ($W_q$) from **93.13 minutes** to **69.10 minutes** — saving **24.03 minutes of idle waiting time per patient** at the hospital's primary administrative bottleneck.

3. **Hospital Target Compliance Gain:**
   > Compliance with the hospital's official 150-minute outpatient stay target increased from **58.15% (15,558 patients)** under the manual system to **62.10% (16,614 patients)** under the digital system. This represents an absolute gain of **+3.95 percentage points**, successfully moving **1,056 additional patients** into compliance with Philippine Heart Center operational standards.

---
_End of Chapter 4 Queue Simulation Notebook._
